Eventually, you'll want to take advantage of the automatization that SCOPE enables. In this tutorial, we will prepare a more advanced workflow:

In [1]:
import os
import Scope

In [2]:
## Path of the data folder. It should be "os.path.abspath('.')+'/Data/Tutorial_4/"
tutorial_folder = os.path.abspath('.')+'/Data/Tutorial_5/'

# Example:

## 1.1) Create Systems: Water, Benzene, Formaldehyde

In [3]:
## In this example, we create three systems, each with 4 .xyz sources.
from Scope.Classes_System import system
water_sys   = system("water")
benz_sys    = system("benzene")
form_sys    = system("formaldehyde")

## We set the paths. See Tutorial_4 for more options and information
for sys in [water_sys, benz_sys, form_sys]:
    sys.sources_path = f"{tutorial_folder}Sources/{sys.name}/"
    sys.calcs_path   = f"{tutorial_folder}Calcs/{sys.name}/"
    sys.sys_path     = f"{tutorial_folder}Systems/{sys.name}/"
    sys.sys_file     = f"{tutorial_folder}Systems/{sys.name}/{sys.name}.npy"
    print(sys)

---------------------------------
   >>> SCOPE System Object >>>   
---------------------------------
 Version               = 1.0
 Type                  = system
 Subtype               = system
 Name                  = water
 Source Path           = /Users/sergivela/Documents/SCOPE/Program/Scope_New/tutorials/Data/Tutorial_5/Sources/water/
 Calculations Path     = /Users/sergivela/Documents/SCOPE/Program/Scope_New/tutorials/Data/Tutorial_5/Calcs/water/
 System File Path      = /Users/sergivela/Documents/SCOPE/Program/Scope_New/tutorials/Data/Tutorial_5/Systems/water/


---------------------------------
   >>> SCOPE System Object >>>   
---------------------------------
 Version               = 1.0
 Type                  = system
 Subtype               = system
 Name                  = benzene
 Source Path           = /Users/sergivela/Documents/SCOPE/Program/Scope_New/tutorials/Data/Tutorial_5/Sources/benzene/
 Calculations Path     = /Users/sergivela/Documents/SCOPE/Program/Scope_New/

## 1.2) Import Molecules in Source Folder 

In [4]:
# Currently, SCOPE is designed so each system has its own folder with sources. In this example, the sources are .xyz files
# Unfortunately, there is no universal way of creating systems. Each problem faces its own system structure. 
# In the SCO add-on, the original data is in a cell2mol CELL format, so a new function had to be created to import molecules from there. 
# New add-ons are being prepared, each one with its own form of importing systems, and arranging molecules within those systems.
for sys in [water_sys, benz_sys, form_sys]:
    sys.load_multiple_xyz(sys.sources_path)
    print(sys)

---------------------------------
   >>> SCOPE System Object >>>   
---------------------------------
 Version               = 1.0
 Type                  = system
 Subtype               = system
 Name                  = water
 Source Path           = /Users/sergivela/Documents/SCOPE/Program/Scope_New/tutorials/Data/Tutorial_5/Sources/water/
 Calculations Path     = /Users/sergivela/Documents/SCOPE/Program/Scope_New/tutorials/Data/Tutorial_5/Calcs/water/
 System File Path      = /Users/sergivela/Documents/SCOPE/Program/Scope_New/tutorials/Data/Tutorial_5/Systems/water/

 # of Sources          = 4
     idx, type, name, formula               
     0: specie geom1 H2-O 
     1: specie geom2 H2-O 
     2: specie geom3 H2-O 
     3: specie geom4 H2-O 


---------------------------------
   >>> SCOPE System Object >>>   
---------------------------------
 Version               = 1.0
 Type                  = system
 Subtype               = system
 Name                  = benzene
 Source Path  

## 2) Create a Job File - Single point with PBE

In [5]:
from Scope.Classes_Input import *

In [6]:
## Job Specs can be created from a file, or from a string. 
## This is an example input read as a string. We will soon go through the different parts
## First, notice that there are four sections, initiated with '&', and terminated with '/'.
## Sections can be empty as in the example below (see &options) or even absent. In those cases, default values will be added
## Also, notice that the values are either introduced as strings 'X', lists [], booleans, or integers/floats...
## Please respect the notation used for each type of variable

test_input = """
&environment
   max_jobs         = 12
   max_procs        = 20
   requested_procs  = 1
/

&options
/

&job_data
   branch       = 'Isolated'
   recipe       = 'all'
   job_setup    = 'regular'
   suffix       = 'scf'
   keyword      = 'pbe scf'

   istate       = 'initial'
   fstate       = 'initial'

   hierarchy    = 1
   requisites   = []
   constrains   = ['self']
   must_be_good = False
/

&qc_data
   software     = 'g16'
   jobtype      = 'scf'
   functional   = 'pbe'
   basis        = 'sto-3g'
   is_grimme    = True
/"""

In [7]:
# IMPORTANT. Notice that, when running the 'scope_run_job' script. The job file (-j arg) must be an actual file, not a string as in the cell above
# Thus, we save this text to an actual file, so we can call it later
from Scope.Read_Write import save_text
file_name = ''.join([tutorial_folder,"test_job.job"])
save_text(test_input, file_name)